# Hito 3 · NLP aplicado a anuncios de venta

David Díaz Vílchez

> **Versión para portfolio.** Trabajo académico de David Díaz Vílchez. Las salidas conservadas proceden de la entrega original; no se han reejecutado los entrenamientos al preparar el repositorio. Se han retirado muestras de anuncios, metadatos de sesión y salidas de infraestructura. Consulta `../docs/resultados.md` y `../docs/reproduccion.md`.


In [ ]:
# Dependencias: instalar requirements-nlp.txt antes de ejecutar este cuaderno.

In [ ]:
import re, string, numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVR
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

print("OK - entorno listo")

In [ ]:
# cargamos el dataset previamente descargado e importado
df = pd.read_csv("../data/Datos.csv")
print(df.shape)
df.head(2)

In [ ]:
# Normalizamos
df.columns = [c.lower() for c in df.columns]
df.columns

In [ ]:
# Se conserva el separador original para no alterar el experimento histórico.
# Creamos la columna de texto final (NLP)
df_nlp = df.copy()

df_nlp["text"] = (
    df_nlp["titulo"].astype(str) + "_toggle_ " +
    df_nlp["tags"].astype(str)
)

df_nlp = df_nlp[["text", "precioactual"]].dropna()

df_nlp.rename(columns={"precioactual": "price"}, inplace=True)

print(df_nlp.shape)
df_nlp.head(3)

In [ ]:
# Limpieza de texto
spanish_stop = set(stopwords.words("spanish"))

def clean_text(t: str) -> str:
    t = t.lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"<.*?>", " ", t)
    t = re.sub(r"\d+", " ", t)
    t = t.translate(str.maketrans("", "", string.punctuation))
    t = re.sub(r"\s+", " ", t).strip()
    return t

df_nlp["text_clean"] = df_nlp["text"].astype(str).apply(clean_text)

# quitamos textos demasiado cortos para reducir el ruido
df_nlp = df_nlp[df_nlp["text_clean"].str.len() >= 20].copy()

X = df_nlp["text_clean"].values
y = df_nlp["price"].astype(float).values

print("Filas finales:", df_nlp.shape)
df_nlp.head(2)

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", len(X_train), "Test:", len(X_test))

Train: 9460 Test: 2366


In [ ]:
# Pipeline + GridSearch
tfidf = TfidfVectorizer(stop_words=list(spanish_stop))

pipe = Pipeline([
    ("tfidf", tfidf),
    ("model", Ridge())
])

param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_df": [0.9, 0.95],
    "model__alpha": [0.5, 1.0, 3.0, 10.0]
}

gs = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1
)

gs.fit(X_train, y_train)

print("Mejores params:", gs.best_params_)
print("Mejor MAE CV:", -gs.best_score_)

Mejores params: {'model__alpha': 1.0, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Mejor MAE CV: 439611.5944795751


In [ ]:
# Evaluación (MAE y MSE)
best_tfidf = gs.best_estimator_

pred = best_tfidf.predict(X_test)

mae_tfidf = mean_absolute_error(y_test, pred)
mse_tfidf = mean_squared_error(y_test, pred)

print("TF-IDF + Ridge -> MAE:", mae_tfidf, "| MSE:", mse_tfidf)

TF-IDF + Ridge -> MAE: 441369.8025535322 | MSE: 732555548646.6615


# MODELO 2: Word2Vec (entrenado) + CNN (Keras) para regresión

In [ ]:
# Tokenizamos y creamos las secuencias
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 30000
max_len = 80

tok = Tokenizer(num_words=max_words, oov_token="<OOV>")
tok.fit_on_texts(X_train)

X_train_seq = pad_sequences(tok.texts_to_sequences(X_train), maxlen=max_len, padding="post", truncating="post")
X_test_seq  = pad_sequences(tok.texts_to_sequences(X_test),  maxlen=max_len, padding="post", truncating="post")

word_index = tok.word_index
vocab_size = min(max_words, len(word_index) + 1)

print("vocab_size:", vocab_size)
print("X_train_seq:", X_train_seq.shape, "X_test_seq:", X_test_seq.shape)

vocab_size: 9383
X_train_seq: (9460, 80) X_test_seq: (2366, 80)


In [ ]:
# Entrenamos Word2Vec
from gensim.models import Word2Vec

train_tokens = [t.split() for t in X_train]

w2v_dim = 100
w2v = Word2Vec(
    sentences=train_tokens,
    vector_size=w2v_dim,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

print("Word2Vec entrenado")

Word2Vec entrenado


In [ ]:
# Creamos la matriz de embeddings
embedding_matrix = np.zeros((vocab_size, w2v_dim), dtype=np.float32)

for word, i in word_index.items():
    if i >= vocab_size:
        continue
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]

print("Embedding matrix:", embedding_matrix.shape)

Embedding matrix: (9383, 100)


In [ ]:
# CNN para regresión
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

model_cnn = keras.Sequential([
    keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=w2v_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    keras.layers.Conv1D(128, 5, activation="relu"),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1)
])

model_cnn.compile(optimizer="adam", loss="mse")
model_cnn.summary()

In [ ]:
# Entrenamos y evaluamos (MAE y MSE)
history_cnn = model_cnn.fit(
    X_train_seq, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)

pred_cnn = model_cnn.predict(X_test_seq).ravel()

mae_cnn = mean_absolute_error(y_test, pred_cnn)
mse_cnn = mean_squared_error(y_test, pred_cnn)

print("Word2Vec + CNN -> MAE:", mae_cnn, "| MSE:", mse_cnn)

Word2Vec + CNN -> MAE: 776813.1824281488 | MSE: 1491770785764.991


# MODELO 3: Transformer (fine-tuning) para regresión

In [ ]:
# Dependencias: instalar requirements-nlp.txt antes de ejecutar este cuaderno.

In [ ]:
# Preparamos datasets
from datasets import Dataset

train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df  = pd.DataFrame({"text": X_test,  "label": y_test})

train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

train_ds, test_ds

(Dataset({
     features: ['text', 'label'],
     num_rows: 9460
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 2366
 }))

In [ ]:
# Tokenizamos
from transformers import AutoTokenizer

model_name = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_tok = train_ds.map(tokenize, batched=True)
test_tok  = test_ds.map(tokenize, batched=True)

train_tok = train_tok.remove_columns(["text"])
test_tok  = test_tok.remove_columns(["text"])

train_tok.set_format("torch")
test_tok.set_format("torch")

train_tok[0]

In [ ]:
# Cargamos el modelo para la regresión
from transformers import AutoModelForSequenceClassification

model_tr = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)

In [ ]:
# Métricas MAE y MSE
import numpy as np

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    mae = np.mean(np.abs(preds - labels))
    mse = np.mean((preds - labels) ** 2)
    return {"mae": mae, "mse": mse}

In [ ]:
# Entrenamiento
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model_tr,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics
)

trainer.train()
eval_tr = trainer.evaluate()
eval_tr

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 2630934069248.0,
 'eval_mae': 1034972.3125,
 'eval_mse': 2630934331392.0,
 'eval_runtime': 16.4983,
 'eval_samples_per_second': 143.409,
 'eval_steps_per_second': 17.941,
 'epoch': 2.0}

# Conclusión
## Modelo Transformer (BERT)

Se ha realizado fine-tuning de un modelo Transformer preentrenado en español para una tarea de regresión del precio de la vivienda a partir del texto.

Aunque el modelo es capaz de aprender patrones semánticos complejos, los
resultados obtenidos son inferiores a los del modelo TF-IDF. Esto puede deberse a textos cortos, escalas elevadas del precio o el tamño del dataset.

Este experimento demuestra que modelos más complejos no siempre garantizan
mejor rendimiento y que enfoques mas basicos y clásicos bien ajustados pueden ser más efectivos.
